In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train = pd.read_csv('/content/train.csv')
train.head(5)

/tmp/ipykernel_16327/3756571849.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('/content/train.csv')


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [ ]:
store = pd.read_csv('/content/store.csv')
store.head(5)

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0.0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1.0,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1.0,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0.0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0.0,NaN,NaN,NaN


In [ ]:
train['Date'] = pd.to_datetime(train['Date'])

In [ ]:
df = train.merge(store, on = 'Store', how = 'left')
df = df[df['Open'] == 1].copy()

In [ ]:
df['StateHoliday'] = df['StateHoliday'].replace('0', 0)
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(
    df['CompetitionDistance'].max() * 2)
df['Assortment'] = df['Assortment'].fillna(df['Assortment'].mode()[0])
df['Promo2'] = df['Promo2'].fillna(0)

df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week']  = df['Date'].dt.isocalendar().week.astype(int)

print(f"Base df shape: {df.shape}")

Base df shape: (844392, 21)


In [ ]:
drop_cols = ['Customers', 'Open', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear']

df = df.drop(columns=drop_cols)
print(f"Shape after dropping: {df.shape}")
print(f"Remaining nulls:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

Shape after dropping: (844392, 14)
Remaining nulls:
StoreType    566750
dtype: int64


In [ ]:
df['IsHoliday'] = df['StateHoliday'].apply(lambda x: 0 if x == 0 else 1)
df = df.drop(columns=['StateHoliday'])

storetype_map = {'a':0, 'b':1, 'c':2}
df['StoreType'] = df['StoreType'].map(storetype_map)

In [ ]:
assortment_map = {'a':0, 'b':1, 'c':2}
df['Assortment'] = df['Assortment'].map(assortment_map)

print('Encoding done')
print(df[['StoreType', 'Assortment', 'IsHoliday']].value_counts())

Encoding done
StoreType  Assortment  IsHoliday
0.0        0           0            98866
           2           0            53006
2.0        0           0            15122
           2           0            13796
1.0        0           0             2749
           1           0             2709
                       1               86
           0           1               77
0.0        2           1               46
           0           1               35
2.0        2           1                5
           0           1                3
Name: count, dtype: int64


In [ ]:
df['DayOfMonth'] = df['Date'].dt.day
df['Quarter']    = df['Date'].dt.quarter
df['IsWeekend']  = df['DayOfWeek'].apply(lambda x: 1 if x >= 6 else 0)

# Days to and from month end/start (useful for retail patterns)
df['DaysTillMonthEnd'] = df['Date'].dt.days_in_month - df['Date'].dt.day
df['IsMonthStart']     = (df['Date'].dt.day <= 3).astype(int)
df['IsMonthEnd']       = (df['Date'].dt.day >= 28).astype(int)

print("Date features added")
print(df[['DayOfMonth', 'Quarter', 'IsWeekend',
          'DaysTillMonthEnd', 'IsMonthStart', 'IsMonthEnd']].head())

Date features added
   DayOfMonth  Quarter  IsWeekend  DaysTillMonthEnd  IsMonthStart  IsMonthEnd
0          31        3          0                 0             0           1
1          31        3          0                 0             0           1
2          31        3          0                 0             0           1
3          31        3          0                 0             0           1
4          31        3          0                 0             0           1


In [ ]:
# Sort first — essential for lag features
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

# Lag features — past sales at same store
lag_days = [1, 7, 14, 28]
for lag in lag_days:
    df[f'lag_{lag}'] = df.groupby('Store')['Sales'].shift(lag)

# Rolling statistics — per store
windows = [7, 14, 28]
for w in windows:
    df[f'rolling_mean_{w}'] = df.groupby('Store')['Sales'].transform(
        lambda x: x.shift(1).rolling(w, min_periods=1).mean())
    df[f'rolling_std_{w}'] = df.groupby('Store')['Sales'].transform(
        lambda x: x.shift(1).rolling(w, min_periods=1).std())
    df[f'rolling_max_{w}'] = df.groupby('Store')['Sales'].transform(
        lambda x: x.shift(1).rolling(w, min_periods=1).max())

print("Lag and rolling features added")
print(f"Shape: {df.shape}")
print(df[['Store', 'Date', 'Sales', 'lag_7',
          'rolling_mean_7', 'rolling_std_7']].head(20))

Lag and rolling features added
Shape: (844392, 33)
    Store       Date  Sales   lag_7  rolling_mean_7  rolling_std_7
0       1 2013-01-02   5530     NaN             NaN            NaN
1       1 2013-01-03   4327     NaN     5530.000000            NaN
2       1 2013-01-04   4486     NaN     4928.500000     850.649458
3       1 2013-01-05   4997     NaN     4781.000000     653.506695
4       1 2013-01-07   7176     NaN     4835.000000     544.406098
5       1 2013-01-08   5580     NaN     5303.200000    1148.189749
6       1 2013-01-09   5471     NaN     5349.333333    1033.170589
7       1 2013-01-10   4892  5530.0     5366.714286     944.271803
8       1 2013-01-11   4881  4327.0     5275.571429     956.594456
9       1 2013-01-12   4952  4486.0     5354.714286     885.295754
10      1 2013-01-14   4717  4997.0     5421.285714     824.518388
11      1 2013-01-15   3900  7176.0     5381.285714     854.769309
12      1 2013-01-16   4008  5580.0     4913.285714     551.335262
13      1 2

In [ ]:
split_date = '2015-06-15'

train_df = df[df['Date'] <  split_date].copy()
test_df  = df[df['Date'] >= split_date].copy()

train_df = train_df.dropna(subset=[f'lag_{l}' for l in lag_days])
test_df  = test_df.dropna(subset=[f'lag_{l}' for l in lag_days])

print(f"Train: {train_df.shape} | {train_df['Date'].min()} to {train_df['Date'].max()}")
print(f"Test:  {test_df.shape}  | {test_df['Date'].min()} to {test_df['Date'].max()}")
print(f"\nTrain stores: {train_df['Store'].nunique()}")
print(f"Test stores:  {test_df['Store'].nunique()}")

Train: (767320, 33) | 2013-01-29 00:00:00 to 2015-06-14 00:00:00
Test:  (45852, 33)  | 2015-06-15 00:00:00 to 2015-07-31 00:00:00

Train stores: 1115
Test stores:  1115


In [ ]:
store1_df = df[df['Store'] == 1][['Date', 'Sales', 'Promo',
                                   'IsHoliday', 'SchoolHoliday']].copy()
store1_df = store1_df.sort_values('Date').set_index('Date')

date_range = pd.date_range(store1_df.index.min(),
                            store1_df.index.max(), freq='D')
missing_dates = date_range.difference(store1_df.index)
print(f"Store 1 records: {len(store1_df)}")
print(f"Missing dates (closed days): {len(missing_dates)}")
print(store1_df.head(10))

Store 1 records: 781
Missing dates (closed days): 160
            Sales  Promo  IsHoliday  SchoolHoliday
Date                                              
2013-01-02   5530      0          0              1
2013-01-03   4327      0          0              1
2013-01-04   4486      0          0              1
2013-01-05   4997      0          0              1
2013-01-07   7176      1          0              1
2013-01-08   5580      1          0              1
2013-01-09   5471      1          0              1
2013-01-10   4892      1          0              1
2013-01-11   4881      1          0              1
2013-01-12   4952      0          0              0


In [ ]:
# Full dataset for XGBoost
train_df.to_csv('train_processed.csv', index=False)
test_df.to_csv('test_processed.csv', index=False)

# Store 1 for ARIMA and Prophet
store1_df.to_csv('store1_timeseries.csv')

print("Files saved:")
print(f"  train_processed.csv  — {train_df.shape}")
print(f"  test_processed.csv   — {test_df.shape}")
print(f"  store1_timeseries.csv — {store1_df.shape}")

Files saved:
  train_processed.csv  — (767320, 33)
  test_processed.csv   — (45852, 33)
  store1_timeseries.csv — (781, 4)


In [ ]:
print("=== FINAL SUMMARY ===")
print(f"\nTrain date range : {train_df['Date'].min()} to {train_df['Date'].max()}")
print(f"Test date range  : {test_df['Date'].min()} to {test_df['Date'].max()}")
print(f"\nFeatures available for XGBoost:")
feature_cols = [c for c in train_df.columns
                if c not in ['Date', 'Sales', 'YearMonth']]
for col in feature_cols:
    print(f"  {col}")
print(f"\nTotal features: {len(feature_cols)}")

=== FINAL SUMMARY ===

Train date range : 2013-01-29 00:00:00 to 2015-06-14 00:00:00
Test date range  : 2015-06-15 00:00:00 to 2015-07-31 00:00:00

Features available for XGBoost:
  Store
  DayOfWeek
  Promo
  SchoolHoliday
  StoreType
  Assortment
  CompetitionDistance
  Promo2
  Year
  Month
  Week
  IsHoliday
  DayOfMonth
  Quarter
  IsWeekend
  DaysTillMonthEnd
  IsMonthStart
  IsMonthEnd
  lag_1
  lag_7
  lag_14
  lag_28
  rolling_mean_7
  rolling_std_7
  rolling_max_7
  rolling_mean_14
  rolling_std_14
  rolling_max_14
  rolling_mean_28
  rolling_std_28
  rolling_max_28

Total features: 31
